In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.is_available()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Total: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)} GB")
    device = torch.device("cuda:0")
else:
    print("No GPU available, training on CPU.")
    device = torch.device("cpu")


GPU is available: True
GPU Name: NVIDIA RTX 6000 Ada Generation
GPU Memory Total: 47.99 GB


In [2]:
# !pip install surfa trimesh nibabel numpy torch

In [3]:
# # import torch
# TORCH = torch.__version__.split('+')[0]
# CUDA = 'cu' + torch.version.cuda.replace('.', '')

# # 2. Install torch_scatter via the official wheel
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html


In [4]:
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
# !pip install torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

In [5]:
import torch
import numpy as np
import surfa as sf
import os
from topofit.topofit import ico, io
from model import AttentionBasedEncoder # <-- Updated Class Name!
from layers.mesh import Mesh # Import YOUR custom MeshCNN object!
from Losses import BrainRegistrationLoss
import torch.optim as optim

In [6]:
class DummyOpt:
    """MeshCNN requires an options object to initialize. We provide a dummy one here."""
    num_aug = 1
    # Add any other default MeshCNN arguments your mesh.py might check for

def convert_to_meshcnn(surfa_template, save_path="clean_lh.obj"):
    """Converts a surfa.Mesh into a layers.mesh.Mesh by bridging via .obj format"""
    print("Building MeshCNN Topological Edges (This takes a moment...)")

    # 1. Write the surfa vertices and faces to an .obj file
    with open(save_path, 'w') as f:
        for v in surfa_template.vertices:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for face in surfa_template.faces:
            # .obj files are 1-indexed, so we add 1 to the face indices
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")

    # 2. Load it into your custom MeshCNN object so it builds the gemm_edges
    opt = DummyOpt()
    meshcnn_obj = Mesh(file=save_path, opt=opt)
    return meshcnn_obj

# --- 4. The Left Hemisphere Loader ---
def load_left_hemisphere(subj_dir):
    print("Loading MRI and aligning Left Hemisphere template...")
    image = sf.load_volume(f'{subj_dir}/mri/norm.mgz')
    affine = sf.load_affine(f'{subj_dir}/mri/transforms/talairach.xfm.lta').inv().convert(space='vox', target=image)

    # Get the TopoFit template
    template = ico.get_initial_template('lh')
    template.vertices = affine.transform(template.vertices)

    crop_idx = io.compute_image_cropping(image.baseshape, template.vertices)
    cropped_img = image[crop_idx].reshape((96, 144, 192))
    cropped_img = cropped_img.astype(np.float32) / cropped_img.percentile(99.99, nonzero=True)

    verts = template.convert(space='vox', geometry=cropped_img).vertices.astype(np.float32)
    template.vertices = verts # Update the template with cropped vertices

    # --- NEW: Convert to MeshCNN Object ---
    meshcnn_topology = convert_to_meshcnn(template)

    mri_tensor = torch.tensor(cropped_img.data).unsqueeze(0).unsqueeze(0)
    vert_tensor = torch.tensor(verts).unsqueeze(0)

    return mri_tensor, vert_tensor, [meshcnn_topology]

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

subj_path = "/mnt/c/Users/ADMIN/Documents/Dev_Projects/brain mri/VisionEncoder/OASIS_0001"
mri_batch, vertex_batch, mesh_topology = load_left_hemisphere(subj_path)

mri_batch = mri_batch.to(device, dtype=torch.float32)
vertex_batch = vertex_batch.to(device, dtype=torch.float32)

print(f"Input MRI Shape: {mri_batch.shape}")
print(f"Input Vertex Shape: {vertex_batch.shape}")

print("\nInitializing Architecture...")
model = AttentionBasedEncoder().to(device) # <-- Updated Class Name!

print("Running Forward Pass (with Deformation Graph!)...")
with torch.no_grad():
    # Now it only catches the single Deformed Mesh output!
    deformed_mesh = model(mri_batch, vertex_batch, mesh_topology)

print(f"\nSUCCESS! Deformed Brain Surface Shape: {deformed_mesh.shape}")

Using Device: cuda
Loading MRI and aligning Left Hemisphere template...
Building MeshCNN Topological Edges (This takes a moment...)
Input MRI Shape: torch.Size([1, 1, 96, 144, 192])
Input Vertex Shape: torch.Size([1, 40962, 3])

Initializing Architecture...
Running Forward Pass (with Deformation Graph!)...

SUCCESS! Deformed Brain Surface Shape: torch.Size([1, 40962, 3])


In [9]:
import torch
import torch.optim as optim
import surfa as sf
from topofit.topofit import ico
from Losses import BrainRegistrationLoss

# 1. Setup Optimizer
learning_rate = 1e-4
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

first_layer_weights_initial = list(model.parameters())[0].clone().detach()
last_layer_weights_initial = list(model.parameters())[-1].clone().detach()

# --- LOAD ACTUAL GROUND TRUTH ---
gt_mesh_path = "/mnt/c/Users/ADMIN/Documents/Dev_Projects/brain mri/VisionEncoder/OASIS_0001/surf/lh.white"
print(f"Loading Ground Truth Mesh (107,418 vertices) from:\n{gt_mesh_path}\n")
gt_mesh = sf.load_mesh(gt_mesh_path)

# 2. Initialize the Loss Function
# Send the 107k target vertices to the GPU
gt_vertices = torch.tensor(gt_mesh.vertices, dtype=torch.float32).unsqueeze(0).to(device)

# We grab the 40k edges directly from the template!
edges_array = mesh_topology[0].edges 

criterion = BrainRegistrationLoss(
    edges=edges_array, 
    w_chamfer=1.0, 
    w_edge=0.1
).to(device)

print("Starting Dummy Training Step...\n")

# --- THE TRAINING LOOP (1 Step) ---
optimizer.zero_grad() 

print("1. Running Forward Pass...")
deformed_mesh = model(mri_batch, vertex_batch, mesh_topology)

print("2. Calculating Chamfer Loss...")
total_loss, chamfer, edge = criterion(deformed_mesh, gt_vertices)

print("3. Running Backward Pass (Calculating Gradients!)...")
total_loss.backward()

print("4. Optimizing (Updating Weights!)...")
optimizer.step()
# ----------------------------------

first_layer_weights_new = list(model.parameters())[0].clone().detach()
last_layer_weights_new = list(model.parameters())[-1].clone().detach()

cnn_changed = not torch.equal(first_layer_weights_initial, first_layer_weights_new)
decoder_changed = not torch.equal(last_layer_weights_initial, last_layer_weights_new)

print("\n" + "=" * 50)
print("             TRAINING STEP RESULTS")
print("=" * 50)
print(f"Total Loss: {total_loss.item():.4f}")
print(f" -> Chamfer Distance: {chamfer.item():.4f}")
print(f" -> Edge Strain Loss: {edge.item():.4f}\n")

print(f"CNN Weights Updated?       {'✅ YES' if cnn_changed else '❌ NO'}")
print(f"Decoder Weights Updated?   {'✅ YES' if decoder_changed else '❌ NO'}")
print("=" * 50)

if cnn_changed and decoder_changed:
    print("\n🎉 SUCCESS! Gradients flowing perfectly with Chamfer Loss!")

Loading Ground Truth Mesh (107,418 vertices) from:
/mnt/c/Users/ADMIN/Documents/Dev_Projects/brain mri/VisionEncoder/OASIS_0001/surf/lh.white

Starting Dummy Training Step...

1. Running Forward Pass...
2. Calculating Chamfer Loss...
3. Running Backward Pass (Calculating Gradients!)...
4. Optimizing (Updating Weights!)...

             TRAINING STEP RESULTS
Total Loss: 175.6959
 -> Chamfer Distance: 175.6864
 -> Edge Strain Loss: 0.0954

CNN Weights Updated?       ✅ YES
Decoder Weights Updated?   ✅ YES

🎉 SUCCESS! Gradients flowing perfectly with Chamfer Loss!


In [10]:
import time

# We will run 10 training steps (epochs) on this single MRI to watch it learn
epochs = 10

print("==================================================")
print("      STARTING 10-STEP LEARNING VERIFICATION")
print("==================================================")

for epoch in range(epochs):
    start_time = time.time()
    
    # 1. Clear old gradients
    optimizer.zero_grad() 
    
    # 2. Forward Pass
    deformed_mesh = model(mri_batch, vertex_batch, mesh_topology)
    
    # 3. Calculate Loss
    total_loss, chamfer, edge = criterion(deformed_mesh, gt_vertices)
    
    # 4. Backward Pass (Calculate Gradients)
    total_loss.backward()
    
    # 5. Optimize (Update Weights)
    optimizer.step()
    
    step_time = time.time() - start_time
    
    # Print the progress of each step
    print(f"Step {epoch+1:02d}/{epochs} | Loss: {total_loss.item():.4f} | (Chamfer: {chamfer.item():.2f}, Edge: {edge.item():.4f}) | Time: {step_time:.2f}s")

print("==================================================")
print("✅ Verification Complete!")
print("Look at the Loss column. Did the number go down from step 1 to step 10?")

      STARTING 10-STEP LEARNING VERIFICATION
Step 01/10 | Loss: 175.3728 | (Chamfer: 175.36, Edge: 0.0954) | Time: 6.23s
Step 02/10 | Loss: 175.1272 | (Chamfer: 175.12, Edge: 0.0955) | Time: 4.94s
Step 03/10 | Loss: 174.9285 | (Chamfer: 174.92, Edge: 0.0955) | Time: 4.91s
Step 04/10 | Loss: 174.7269 | (Chamfer: 174.72, Edge: 0.0956) | Time: 4.87s
Step 05/10 | Loss: 174.5280 | (Chamfer: 174.52, Edge: 0.0957) | Time: 4.91s
Step 06/10 | Loss: 174.3347 | (Chamfer: 174.33, Edge: 0.0958) | Time: 4.85s
Step 07/10 | Loss: 174.1447 | (Chamfer: 174.14, Edge: 0.0959) | Time: 4.91s
Step 08/10 | Loss: 173.9604 | (Chamfer: 173.95, Edge: 0.0960) | Time: 4.91s
Step 09/10 | Loss: 173.7853 | (Chamfer: 173.78, Edge: 0.0961) | Time: 4.99s
Step 10/10 | Loss: 173.6169 | (Chamfer: 173.61, Edge: 0.0962) | Time: 4.95s
✅ Verification Complete!
Look at the Loss column. Did the number go down from step 1 to step 10?


In [11]:
import time

# We will run 10 training steps (epochs) on this single MRI to watch it learn
epochs = 30

print("==================================================")
print("      STARTING 30-STEP LEARNING VERIFICATION")
print("==================================================")

for epoch in range(epochs):
    start_time = time.time()
    
    # 1. Clear old gradients
    optimizer.zero_grad() 
    
    # 2. Forward Pass
    deformed_mesh = model(mri_batch, vertex_batch, mesh_topology)
    
    # 3. Calculate Loss
    total_loss, chamfer, edge = criterion(deformed_mesh, gt_vertices)
    
    # 4. Backward Pass (Calculate Gradients)
    total_loss.backward()
    
    # 5. Optimize (Update Weights)
    optimizer.step()
    
    step_time = time.time() - start_time
    
    # Print the progress of each step
    print(f"Step {epoch+1:02d}/{epochs} | Loss: {total_loss.item():.4f} | (Chamfer: {chamfer.item():.2f}, Edge: {edge.item():.4f}) | Time: {step_time:.2f}s")

print("==================================================")
print("✅ Verification Complete!")
print("Look at the Loss column. Did the number go down from step 1 to step 10?")

      STARTING 30-STEP LEARNING VERIFICATION
Step 01/30 | Loss: 173.4450 | (Chamfer: 173.44, Edge: 0.0963) | Time: 5.20s
Step 02/30 | Loss: 173.2681 | (Chamfer: 173.26, Edge: 0.0964) | Time: 4.93s
Step 03/30 | Loss: 173.0882 | (Chamfer: 173.08, Edge: 0.0964) | Time: 4.78s
Step 04/30 | Loss: 172.9065 | (Chamfer: 172.90, Edge: 0.0965) | Time: 5.02s
Step 05/30 | Loss: 172.7234 | (Chamfer: 172.71, Edge: 0.0966) | Time: 4.93s
Step 06/30 | Loss: 172.5387 | (Chamfer: 172.53, Edge: 0.0966) | Time: 5.02s
Step 07/30 | Loss: 172.3516 | (Chamfer: 172.34, Edge: 0.0966) | Time: 4.83s
Step 08/30 | Loss: 172.1615 | (Chamfer: 172.15, Edge: 0.0966) | Time: 4.88s
Step 09/30 | Loss: 171.9684 | (Chamfer: 171.96, Edge: 0.0966) | Time: 4.94s
Step 10/30 | Loss: 171.7720 | (Chamfer: 171.76, Edge: 0.0966) | Time: 5.26s
Step 11/30 | Loss: 171.5720 | (Chamfer: 171.56, Edge: 0.0966) | Time: 4.75s
Step 12/30 | Loss: 171.3687 | (Chamfer: 171.36, Edge: 0.0966) | Time: 4.77s
Step 13/30 | Loss: 171.1626 | (Chamfer: 171